## EX1 - 与 TOES 的对比实验 (Benchmark Q1)

### 定义 TOES 求解器接口 (Wrapper)

In [4]:
# 封装 TOES 求解器 (不依赖 ROS)
import sys, os 
import numpy as np
import time
proj_path = r'/home/songxy/code/time-optimal-ergodic-search'
sys.path.insert(0, os.path.abspath(proj_path))

# Cell 01: Mock ROS modules (修正版：移除错误的 __getattr__ 覆盖)
from unittest.mock import MagicMock

# MagicMock 默认就支持无限链式调用 (m.A.B.C)，不需要额外配置
class MockObject(MagicMock):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # 关键修正：移除了 self.__getattr__ = ... 那一行
        # 甚至可以完全不写 __init__，直接用 pass

# 1. 伪装 rospy
sys.modules["rospy"] = MockObject()

# 2. 伪装消息类型
# 只要代码里有 "from X import Y" 或 "import X"，X 就必须在这里注册
sys.modules["grid_map_msgs"] = MockObject()
sys.modules["grid_map_msgs.msg"] = MockObject()
sys.modules["std_msgs"] = MockObject()
sys.modules["std_msgs.msg"] = MockObject()
sys.modules["geometry_msgs"] = MockObject()
sys.modules["geometry_msgs.msg"] = MockObject()
sys.modules["visualization_msgs"] = MockObject()
sys.modules["visualization_msgs.msg"] = MockObject()
sys.modules["drone_env_viz"] = MockObject()
sys.modules["drone_env_viz.msg"] = MockObject()
sys.modules["tf"] = MockObject()

print("✅ ROS 模块已伪装完成 (MagicMock)，可以导入 TOES 库了。")
# 假设这些库在你的环境中可用

from experiments.bias_search.target_distribution import TargetDistribution
from experiments.bias_search.build_solver import build_erg_time_opt_solver

class TOESSolver:
    def __init__(self, workspace_bounds=np.array([[0., 3.5], [-1., 3.5]])):
        self.workspace_bounds = workspace_bounds

    def solve(self, start, goal, distribution_manager, gamma, max_iter=2000):
        """
        start: [x, y]
        goal: [x, y]
        distribution_manager: TargetDistribution 对象 (包含 GMM)
        gamma: ergodic upper bound
        """
        # 构造 TOES 需要的 args
        # 注意：TOES 使用 DoubleIntegrator，状态是 [x, y, vx, vy]
        x0 = np.array([start[0], start[1], 0., 0.])
        xf = np.array([goal[0], goal[1], 0., 0.])
        
        args = {
            'N': 100,  # 离散点数
            'x0': x0,
            'xf': xf,
            'erg_ub': gamma, # 关键参数
            'alpha': 0.5,
            'wrksp_bnds': self.workspace_bounds
        }
        
        # 构建求解器
        # 注意：这里传入的是 distribution 对象，需确保 build_solver 兼容
        solver, _ = build_erg_time_opt_solver(args, distribution_manager)
        
        # 开始计时
        t0 = time.time()
        # 求解
        solver.solve(args=args, max_iter=max_iter, eps=1e-5)
        solve_time = time.time() - t0
        
        sol = solver.get_solution() # 获取结果 {'x': ..., 'u': ..., 'tf': ...}
        
        # 提取轨迹 (只取 x, y)
        trajectory = sol['x'][:, :2]
        
        return {
            'trajectory': trajectory,
            'time': solve_time,
            'tf': float(sol['tf']), # 物理时间
            'success': True # 假设求解成功，实际可根据 sol 状态判断
        }

✅ ROS 模块已伪装完成 (MagicMock)，可以导入 TOES 库了。


### 定义 Diffusion 模型接口 (Wrapper)

In [5]:
# 封装 Diffusion 模型
import torch

class DiffusionSolver:
    def __init__(self, model, device):
        self.model = model
        self.device = device
        self.config = model.config
        
        # 归一化参数
        self.rs_mean = torch.as_tensor(self.config.normalizer.robot_state.mean, device=device)
        self.rs_std = torch.as_tensor(self.config.normalizer.robot_state.std, device=device)

    def solve(self, start, distribution_grid, gaussian_packed, gaussian_mask, gamma):
        """
        start: [x, y]
        distribution_grid: [1, H, W] tensor (热力图)
        gaussian_packed: [1, N, 7] tensor
        gaussian_mask: [1, N] tensor
        gamma: float
        """
        # 1. 准备 Robot State (归一化)
        # 假设初始速度为 0
        rs_phys = torch.tensor([start[0], start[1], 0., 0.], device=self.device).unsqueeze(0)
        rs_norm = (rs_phys - self.rs_mean) / self.rs_std
        
        # 2. 准备 Gamma
        gamma_tensor = torch.tensor([gamma], device=self.device, dtype=torch.float32).view(1, 1)
        
        # 3. 构造输入
        inputs = {
            "distribution": distribution_grid.unsqueeze(0).to(self.device), # [1, 1, H, W]
            "robot_state": rs_norm,
            "gaussian_packed": gaussian_packed.unsqueeze(0).to(self.device),
            "gaussian_padding_mask": gaussian_mask.unsqueeze(0).to(self.device),
            "gamma": gamma_tensor
        }
        
        # 4. 推理 (包含时间统计)
        torch.cuda.synchronize()
        t0 = time.time()
        
        with torch.no_grad():
            # 这里可以加入 xt-corrector 引导逻辑，或者先测 Base
            out = self.model.inference(inputs) 
            
        torch.cuda.synchronize()
        solve_time = time.time() - t0
        
        pred_traj = out["prediction"].squeeze(0).cpu().numpy() # [101, 4]
        
        return {
            'trajectory': pred_traj[:, :2],
            'time': solve_time,
            'success': True
        }

### 主实验循环 (Benchmark Loop)

In [6]:
# 实验主循环
import matplotlib.pyplot as plt

# 1. 初始化
toes_solver = TOESSolver()
diffusion_solver = DiffusionSolver(model, device) # 假设 model 已加载好

# 2. 设置实验条件
# 随便拿一个测试样本
sample = val_loader.dataset[0]
dist_manager = TargetDistribution(...) # 需要根据 sample['gaussian_params'] 重建这个对象供 TOES 用
gamma_levels = [0.2, 0.1, 0.05, 0.01, 0.001]

results = {'toes': [], 'diffusion': []}

# 3. 循环对比
for g in gamma_levels:
    print(f"--- Testing Gamma = {g} ---")
    
    # A. 运行 TOES
    res_toes = toes_solver.solve(start, end, dist_manager, gamma=g)
    # 计算真实的 Ergodic Metric (使用统一的 metric 对象)
    e_toes = metric.energy(torch.tensor(res_toes['trajectory']).unsqueeze(0)...).item()
    
    # B. 运行 Diffusion
    # (注意：这里需要把 sample 的数据拆解传进去)
    res_diff = diffusion_solver.solve(..., gamma=g)
    e_diff = metric.energy(torch.tensor(res_diff['trajectory']).unsqueeze(0)...).item()
    
    # 记录数据
    results['toes'].append({'gamma': g, 'time': res_toes['time'], 'erg': e_toes})
    results['diffusion'].append({'gamma': g, 'time': res_diff['time'], 'erg': e_diff})

# 4. 画图 (复现 PDF 中的 Pareto 曲线)
# 图 1: E vs Gamma (看谁更听话)
# 图 2: Time vs Gamma (看谁更快)
# 图 3: Time vs E (帕累托前沿：看谁性价比高)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (3536526778.py, line 23)